In [1]:
import sys
sys.path.append('../src')

In [2]:
import pandas as pd
import numpy as np
import pyarrow
from joblib import Parallel, delayed
from feature_extraction import add_state_column
from hybrid_model import HybridModel
from technical_indicator_classifier import TechnicalIndicatorClassifier
from simulation import simulate_trade
from stock_prices import get_stock_prices, split_train_test as split_train_val

In [3]:
symbol = "^JKSE"
rolling_windows = [
    ["2023-01-01", "2023-08-31", "2023-12-31"],
    ["2023-07-01", "2024-02-29", "2024-06-30"],
    ["2024-01-01", "2024-08-31", "2024-12-31"],
]

In [4]:
def train_one_seed_classifier(seed, method, peek, train_df, val_df):
    np.random.seed(seed)
    classifier = TechnicalIndicatorClassifier(model = method, peek = peek)
    classifier.train(train_df)
    val_return = simulate_trade(val_df, classifier, plot = False)
    return val_return

In [5]:
peek_values = [3, 4, 5, 6, 7]
methods = ["GaussianNB"]

results = []

for i in range(len(rolling_windows)):
    start_date, split_date, end_date = rolling_windows[i]
    stock_prices = get_stock_prices(symbol, start_date, end_date)
    add_state_column(stock_prices)
    train_df, val_df = split_train_val(stock_prices, split_date)
    
    for peek in peek_values:
        open_prices_val = val_df["open"].values[20:]
        buy_and_hold_return = open_prices_val[-1] / open_prices_val[1] - 1.002
        
        for method in methods:
            val_return = train_one_seed_classifier(0, method, peek, train_df, val_df)
            results.append({
                "window": i + 1,
                "seed": 0,
                "method": method,
                "peek": peek,
                "model_return": val_return,
                "buy_and_hold_return": buy_and_hold_return
            })

$^JKSE: possibly delisted; no price data found  (1d 2023-01-01 -> 2023-01-02)


In [6]:
results = pd.DataFrame(results)

results["excess_return"] = (
    results["model_return"] - results["buy_and_hold_return"]
)

# Optional rounding for neat display
results = results.round({
    "model_return": 3,
    "buy_and_hold_return": 3,
    "excess_return": 3
})

summary = (
    results
    .groupby(["method", "peek"])["excess_return"]
    .agg(
        mean_excess = "mean",
        std_excess = "std"
    )
)

summary["score"] = summary["mean_excess"] - summary["std_excess"]

print(summary)
print(summary.loc[(summary["score"] == summary["score"].max())])

                 mean_excess  std_excess     score
method     peek                                   
GaussianNB 3       -0.022333    0.046177 -0.068511
           4       -0.007333    0.038850 -0.046183
           5        0.003333    0.010116 -0.006783
           6       -0.008333    0.026407 -0.034740
           7       -0.007000    0.027839 -0.034839
                 mean_excess  std_excess     score
method     peek                                   
GaussianNB 5        0.003333    0.010116 -0.006783


In [7]:
# Select method and peek based on highest score
method, peek = summary["score"].idxmax()
k_values = [3, 6, 9]
seeds = [0, 1, 2, 3, 4]

In [8]:
def train_hybrid_instance(seed, peek, k, classifier, train_df, val_df):
    np.random.seed(seed)
    hybrid_model = HybridModel(k = k, classifier = classifier)
    stable = hybrid_model.train(train_df, show_log = False)
    trial = 1
    while not stable:
        stable = hybrid_model.train(train_df, show_log = False)
        trial += 1
        if trial == 10:
            print(f"Model failed to stabilize even after 10 trials (peek {peek}, k {k})")
            return None
    val_return = simulate_trade(val_df, hybrid_model, plot = False)
    return val_return

In [9]:
results = []

for i in range(len(rolling_windows)):
    start_date, split_date, end_date = rolling_windows[i]
    stock_prices = get_stock_prices(symbol, start_date, end_date)
    add_state_column(stock_prices)
    train_df, val_df = split_train_val(stock_prices, split_date)

    open_prices_val = val_df["open"].values[20:]
    buy_and_hold_return = open_prices_val[-1] / open_prices_val[1] - 1.002

    classifier = TechnicalIndicatorClassifier(model = method, peek = peek)
    classifier.train(train_df)

    for k in k_values:

        val_returns = Parallel(n_jobs = len(seeds))(
        delayed(train_hybrid_instance)(seed, peek, k, classifier, train_df, val_df) for seed in seeds)
        for j in range(len(seeds)):
            results.append({
                "window": i + 1,
                "seed": seeds[j],
                "method": method,
                "peek": peek,
                "k": k,
                "model_return": val_returns[j],
                "buy_and_hold_return": buy_and_hold_return
            })
                
    val_return = simulate_trade(val_df, classifier, plot = False)

    results.append({
        "window": i + 1,
        "seed": 0,
        "method": method,
        "peek": peek,
        "k": 0,
        "model_return": val_return,
        "buy_and_hold_return": buy_and_hold_return
    })

$^JKSE: possibly delisted; no price data found  (1d 2023-01-01 -> 2023-01-02)


In [10]:
results = pd.DataFrame(results)

results["excess_return"] = (
    results["model_return"] - results["buy_and_hold_return"]
)

# Optional rounding for neat display
results = results.round({
    "model_return": 3,
    "buy_and_hold_return": 3,
    "excess_return": 3
})

summary = (
    results
    .groupby(["method", "peek", "k"])["excess_return"]
    .agg(
        mean_excess = "mean",
        std_excess = "std"
    )
)
summary["score"] = summary["mean_excess"] - summary["std_excess"]

print(summary)
summary.to_parquet(f"params/{symbol[:4]}.parquet", engine = "pyarrow")

                   mean_excess  std_excess     score
method     peek k                                   
GaussianNB 5    0     0.003333    0.010116 -0.006783
                3    -0.001400    0.036268 -0.037668
                6     0.012067    0.022645 -0.010578
                9     0.023467    0.028693 -0.005226


In [11]:
# Best overall
best_all = summary.loc[summary["score"].idxmax()]

# Best excluding k = 0
summary_nonzero_k = summary.loc[summary.index.get_level_values("k") != 0]
best_nonzero_k = summary_nonzero_k.loc[summary_nonzero_k["score"].idxmax()]

print("=== BEST OVERALL (including k = 0) ===")

print(best_all)

print("\n=== BEST WITH k != 0 ===")
print(best_nonzero_k)

=== BEST OVERALL (including k = 0) ===
mean_excess    0.023467
std_excess     0.028693
score         -0.005226
Name: (GaussianNB, 5, 9), dtype: float64

=== BEST WITH k != 0 ===
mean_excess    0.023467
std_excess     0.028693
score         -0.005226
Name: (GaussianNB, 5, 9), dtype: float64
